In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1563_Pusa_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,198.95,357.77,27.73,42.23,69.93,12.60,4.30,1.24,9.62,...,NaN,15.24,2.86,1.00,227.06,0.0,0.0,45.13,982.00,NaN
1,2024-01-02,197.01,354.10,40.19,42.59,82.60,10.94,6.28,1.37,11.37,...,NaN,14.92,4.22,0.67,225.56,0.0,0.0,66.07,981.64,NaN
2,2024-01-03,211.26,365.51,36.78,43.32,79.87,19.05,4.60,1.77,9.36,...,NaN,14.27,9.54,0.57,253.39,0.0,0.0,49.19,981.40,NaN
3,2024-01-04,243.52,445.83,87.87,39.67,127.70,20.61,4.97,1.76,6.26,...,NaN,14.48,12.06,1.21,160.37,0.0,0.0,20.02,981.57,NaN
4,2024-01-05,176.65,353.14,55.19,39.71,94.15,19.03,4.61,1.54,6.11,...,NaN,15.26,10.93,0.80,196.95,0.0,0.0,17.20,981.77,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,131.58,179.25,34.96,53.55,55.91,75.54,7.37,0.44,5.18,...,NaN,14.85,91.45,1.33,164.37,0.0,0.0,8.43,985.84,NaN
362,2024-12-28,91.58,147.82,71.09,53.97,85.67,58.67,9.41,0.99,4.58,...,NaN,15.25,92.20,0.31,207.72,0.0,0.0,14.88,985.70,NaN
363,2024-12-29,96.93,168.73,31.92,38.40,45.55,71.03,7.87,0.58,4.91,...,NaN,14.97,85.22,0.30,272.65,0.0,0.0,60.55,986.47,NaN
364,2024-12-30,99.96,180.92,53.95,42.89,65.87,61.96,8.74,0.71,5.92,...,NaN,13.69,80.99,0.30,275.72,0.0,0.0,55.17,985.98,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         198.95        357.77       27.73        42.23   
1  2024-01-02         197.01        354.10       40.19        42.59   
2  2024-01-03         211.26        365.51       36.78        43.32   
3  2024-01-04         243.52        445.83       87.87        39.67   
4  2024-01-05         176.65        353.14       55.19        39.71   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      69.93        12.60         4.30        1.24           9.62   
1      82.60        10.94         6.28        1.37          11.37   
2      79.87        19.05         4.60        1.77           9.36   
3     127.70        20.61         4.97        1.76           6.26   
4      94.15        19.03         4.61        1.54           6.11   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             2.51             7.02    15.24    2.86      1

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.866134,1.462032,-0.468344,-0.917043,0.039163,-1.335601,-1.028888,0.327027,-0.703480,-0.632812,-1.204417,-1.665079,-2.984466,-0.345802,1.743615,0.0,0.0,-1.405647,0.072049
1,2024-01-02,1.834809,1.425575,-0.099480,-0.900471,0.384545,-1.473122,-0.613423,0.670135,-0.602802,-0.659171,-1.112261,-1.709062,-2.910602,-0.657652,1.678280,0.0,0.0,-1.046508,-0.089399
2,2024-01-03,2.064903,1.538920,-0.200429,-0.866865,0.310125,-0.801255,-0.965939,1.725852,-0.718438,-0.467131,-1.035324,-1.798402,-2.621664,-0.752152,2.890471,0.0,0.0,-1.336015,-0.197031
3,2024-01-04,2.585803,2.336806,1.312032,-1.034893,1.613963,-0.672018,-0.888301,1.699459,-0.896783,-0.399352,-0.972760,-1.769539,-2.484799,-0.147353,-1.161201,0.0,0.0,-1.836306,-0.120792
4,2024-01-05,1.506058,1.416039,0.344578,-1.033051,0.699396,-0.802912,-0.963840,1.118815,-0.905412,-0.523613,-0.756322,-1.662330,-2.546171,-0.534802,0.432114,0.0,0.0,-1.884671,-0.031099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.778315,-0.311357,-0.254308,-0.395929,-0.343020,0.077308,-0.384707,-1.784407,-0.958915,-0.045397,-0.406302,-1.718683,1.827002,-0.033953,-0.986973,0.0,0.0,-2.035084,1.794157
362,2024-12-28,0.132438,-0.623578,0.815279,-0.376594,0.468232,2.481033,0.043348,-0.332796,-0.993434,-0.094348,0.208348,-1.663705,1.867736,-0.997852,0.901223,0.0,0.0,-1.924461,1.731372
363,2024-12-29,0.218824,-0.415861,-0.344304,-1.093357,-0.625432,0.077308,-0.279792,-1.414906,-0.974449,-0.689295,-0.908505,-1.702190,1.488640,-1.007302,-0.276558,0.0,0.0,-1.141181,2.076691
364,2024-12-30,0.267749,-0.294767,0.307869,-0.886660,-0.071512,2.753590,-0.097239,-1.071798,-0.916343,-0.647874,-0.778304,-1.878121,1.258902,-1.007302,-0.276558,0.0,0.0,-1.233453,1.856942


In [10]:
df.to_excel('pusa2024.xlsx', index=False)